In [1]:
from scipy.io import loadmat
import matplotlib.pyplot as plt
import os
import numpy as np
import tqdm
import glob
import re
import torch

In [3]:
data_fold = '/Users/hailey/Desktop/Yale/cell/p5_Cell/'
day_fold_list = sorted(glob.glob(os.path.join(data_fold, '*/')))
day_list = []
for fold in day_fold_list:
    day_list.append(int(re.sub('.*_Day([0-9]+)/', r'\1', fold)))
data = {}
for day, fold in tqdm.tqdm(zip(day_list, day_fold_list), total=len(day_list)):
    for wi, well in tqdm.tqdm(enumerate(['north', 'south']), total=2, leave=False):
        w_fold = os.path.join(fold, f'{wi + 1:02d}_{well.capitalize()}_Well')
        f_fold_list = glob.glob(os.path.join(w_fold, '02_Cell', '*_1min_*/'))
        assert len(f_fold_list) == 1
        f_fold = f_fold_list[0]
        for mod in tqdm.tqdm(['piv', 'pwv'], leave=False):
            f_list = sorted(glob.glob(os.path.join(f_fold, f'{mod}*.mat')),key=lambda f: int(re.search(r'(\d+)', os.path.basename(f)).group()))        
            for file in tqdm.tqdm(f_list, leave=False):
                file_index = int(re.sub(r'.*?([0-9]+)\.mat', r'\1', file))
                data_key = (day, well, mod, file_index)
                data_load = loadmat(file)
                img = data_load[f'{mod}w'] if f'{mod}w' in data_load else data_load[f'{mod}']
                data[data_key] = img

  0%|                                                              | 0/2 [00:00<?, ?it/s]

  0%|                                                              | 0/2 [00:00<?, ?it/s]


  0%|                                                            | 0/120 [00:00<?, ?it/s]


 22%|██████████▊                                       | 26/120 [00:00<00:00, 256.24it/s]


 47%|███████████████████████▎                          | 56/120 [00:00<00:00, 281.59it/s]


 72%|███████████████████████████████████▊              | 86/120 [00:00<00:00, 284.73it/s]


 97%|███████████████████████████████████████████████▎ | 116/120 [00:00<00:00, 288.76it/s]


                                                                                         

 50%|███████████████████████████                           | 1/2 [00:00<00:00,  2.36it/s]


  0%|                                                            | 0/120 [00:00<?, ?it/s]


 29%|██████████████▌                                   | 35/120 [00:00<00:00, 342.

In [4]:
from torch.utils.data import Dataset, DataLoader
class CellDataset(Dataset):
    def __init__(self, data, patch_size, scale, keys=None, day_to_idx=None):
        self.data = data
        if keys is None:
            keys = []
            for k in data.keys():
                key = (k[0], k[1], k[3])
                if key not in keys:
                    keys.append(key)
        self.keys = keys
        self.patch_size = patch_size
        self.scale = scale
        self.day_to_idx = day_to_idx  # dict mapping day -> class index

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        key_piv = (self.keys[idx][0], self.keys[idx][1], 'piv', self.keys[idx][2])
        key_pwv = (self.keys[idx][0], self.keys[idx][1], 'pwv', self.keys[idx][2])
        img_piv = self.data[key_piv] / self.scale[0]
        img_pwv = self.data[key_pwv] / self.scale[1]
        img = np.stack([img_piv, img_pwv], axis=-1)
        x0 = np.random.randint(0, img.shape[1] - self.patch_size - 1)
        y0 = np.random.randint(0, img.shape[0] - self.patch_size - 1)
        x1 = x0 + self.patch_size
        y1 = y0 + self.patch_size
        img_crop = img[y0:y1, x0:x1]
        x = torch.from_numpy(img_crop).float()
        day = self.keys[idx][0]
        label = self.day_to_idx[day] if self.day_to_idx is not None else day
        y = torch.tensor(label).long()
        return x, y

In [5]:
full_dataset = CellDataset(data, patch_size=128, scale=(5000, 200))
all_keys = full_dataset.keys

unique_days = sorted(set(k[0] for k in all_keys))
day_to_idx = {day: i for i, day in enumerate(unique_days)}
print(day_to_idx)

{0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14}


In [7]:
rng = np.random.default_rng(seed=42)
rng.shuffle(unique_days)
n_train_days = int(0.8 * len(unique_days))
train_days = set(unique_days[:n_train_days])
test_days  = set(unique_days[n_train_days:])

train_keys = [k for k in all_keys if k[0] in train_days]
test_keys  = [k for k in all_keys if k[0] not in train_days]
print(f"Train: {len(train_keys)}  Test: {len(test_keys)}")

train_dataset = CellDataset(data, patch_size=128, scale=(5000, 200), keys=train_keys, day_to_idx=day_to_idx)
test_dataset  = CellDataset(data, patch_size=128, scale=(5000, 200), keys=test_keys, day_to_idx=day_to_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)


Train: 2893  Test: 718


In [8]:
import torch.nn as nn
import torch.nn.functional as F
class Net(nn.Module):
    def __init__(self, n_days):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((5, 5))
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, n_days)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [9]:
import torch.optim as optim
net = Net(n_days=15)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.02, momentum=0.9)
n_epochs = 80
for epoch in range(n_epochs):
    running_loss = 0.0
    net.train()
    for i, data in enumerate(train_loader, 0):
        inputs, labels = data
        inputs = inputs.permute(0, 3, 1, 2)
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch {epoch + 1}, avg loss: {running_loss / len(train_loader):.3f}')
print('Finished Training')

Epoch 1, avg loss: 2.590
Epoch 2, avg loss: 2.351
Epoch 3, avg loss: 1.880
Epoch 4, avg loss: 1.742
Epoch 5, avg loss: 1.692
Epoch 6, avg loss: 1.596
Epoch 7, avg loss: 1.486
Epoch 8, avg loss: 1.454
Epoch 9, avg loss: 1.460
Epoch 10, avg loss: 1.387
Epoch 11, avg loss: 1.373
Epoch 12, avg loss: 1.383
Epoch 13, avg loss: 1.255
Epoch 14, avg loss: 1.145
Epoch 15, avg loss: 1.117
Epoch 16, avg loss: 1.157
Epoch 17, avg loss: 1.064
Epoch 18, avg loss: 1.028
Epoch 19, avg loss: 0.954
Epoch 20, avg loss: 0.937
Epoch 21, avg loss: 0.884
Epoch 22, avg loss: 0.855
Epoch 23, avg loss: 0.871
Epoch 24, avg loss: 0.902
Epoch 25, avg loss: 0.816
Epoch 26, avg loss: 0.806
Epoch 27, avg loss: 0.804
Epoch 28, avg loss: 0.795
Epoch 29, avg loss: 0.780
Epoch 30, avg loss: 0.849
Epoch 31, avg loss: 0.790
Epoch 32, avg loss: 0.876
Epoch 33, avg loss: 0.784
Epoch 34, avg loss: 0.744
Epoch 35, avg loss: 0.694
Epoch 36, avg loss: 1.066
Epoch 37, avg loss: 0.812
Epoch 38, avg loss: 0.779
Epoch 39, avg loss: 0

In [10]:
net.eval()
correct = 0
total = 0
labels_all = []
predicted_all = []
with torch.no_grad():
    for data in test_loader:
        inputs, labels = data
        inputs = inputs.permute(0, 3, 1, 2)
        outputs = net(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        labels_all.extend(list(labels.numpy()))
        predicted_all.extend(list(predicted.numpy()))

labels_all = np.array(labels_all)
predicted_all = np.array(predicted_all)

print(f'Test accuracy: {100 * correct / total:.2f}%')

Test accuracy: 0.00%


In [29]:
len(labels_all)

723

In [22]:
predicted

tensor([ 2,  7,  9,  1,  2,  7,  9,  7,  2, 12,  7,  7,  6,  9,  2, 13, 12, 13,
        12])

In [23]:
labels-predicted

tensor([ 0, -7,  5,  0,  0,  0,  5, -3,  1, -9, -6, -6,  0,  0,  0, -7, -6, -1,
        -6])

In [31]:
np.mean(np.abs(labels_all-predicted_all))

np.float64(3.116182572614108)